# Lab 10: Kuwahara Filter
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Có thể tải `image.jpg` vào Files trước khi chạy; nếu không có, notebook dùng ảnh mẫu có sẵn.

Kết quả: ảnh, `metrics.csv`, báo cáo LaTeX và file `outputs/Lab10.zip`.
Thời gian GPU đã warm-up, đồng bộ; không tính truyền dữ liệu. Dùng GPU thật để đo speedup.


In [ ]:
%pip -q install "numba-cuda[cu12]"


In [ ]:
import time
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from numba import cuda, float32, float64

OUT = Path('outputs/Lab10')
OUT.mkdir(parents=True, exist_ok=True)

assert cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
device = cuda.get_current_device()
print('GPU:', device.name)
MAX_THREADS = device.MAX_THREADS_PER_BLOCK
BLOCKS = [b for b in [(8, 8), (16, 16), (32, 16), (32, 32)]
          if b[0] * b[1] <= MAX_THREADS]
REPEATS = 3

def load_image(path='image.jpg'):
    if Path(path).exists():
        a = plt.imread(path)
        if np.issubdtype(a.dtype, np.floating):
            a = np.clip(a * 255, 0, 255).astype(np.uint8)
        return np.asarray(Image.fromarray(a).convert('RGB')).copy()
    # Ảnh mẫu có màu, cạnh và nhiễu; không cần tải file để Run all.
    y, x = np.indices((256, 256))
    a = np.stack((x, y, (x // 32 % 2) * 160 + 40), axis=-1)
    noise = np.random.default_rng(0).integers(-15, 16, a.shape)
    return np.clip(a + noise, 0, 255).astype(np.uint8)

image = load_image()
H, W = image.shape[:2]
gray = (0.299 * image[:, :, 0] + 0.587 * image[:, :, 1]
        + 0.114 * image[:, :, 2]).astype(np.uint8)
print('Ảnh:', image.shape)

def grid(block):
    return ((W + block[0] - 1) // block[0],
            (H + block[1] - 1) // block[1])

def gpu_time(run):
    run()  # Warm-up: không tính biên dịch JIT vào thời gian.
    cuda.synchronize()
    start = time.time()
    for _ in range(REPEATS):
        run()
    cuda.synchronize()
    return (time.time() - start) / REPEATS

def cpu_time(run):
    start = time.time()
    result = run()
    return result, time.time() - start

def show_images(items):
    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4), squeeze=False)
    for ax, (title, a) in zip(axes[0], items):
        ax.imshow(a, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(OUT / 'images.png', dpi=120)
    plt.show()

def save_image(name, a):
    Image.fromarray(a).save(OUT / name)

def save_results(rows, report_name, report_text):
    df = pd.DataFrame(rows)
    df.to_csv(OUT / 'metrics.csv', index=False)
    print(df.to_string(index=False))
    # Bảng LaTeX nhỏ, không cần thêm thư viện dựng báo cáo.
    def tex(v):
        return str(v).replace('_', r'\_')
    lines = [r'\begin{center}', r'\resizebox{\linewidth}{!}{',
             r'\begin{tabular}{' + 'l' * len(df.columns) + '}', r'\hline',
             ' & '.join(tex(c) for c in df.columns) + r' \\', r'\hline']
    for row in df.itertuples(index=False, name=None):
        lines.append(' & '.join(f'{v:.6g}' if isinstance(v, float) else tex(v)
                                for v in row) + r' \\')
    lines += [r'\hline', r'\end{tabular}', '}', r'\end{center}']
    (OUT / 'results.tex').write_text('\n'.join(lines), encoding='utf-8')
    (OUT / report_name).write_text(report_text, encoding='utf-8')
    shutil.make_archive(str(OUT), 'zip', OUT)
    print('Kết quả và báo cáo:', str(OUT) + '.zip (tải trong Files bên trái)')


In [ ]:
@cuda.jit
def rgb2hsv(src, hue, sat, val):
    x, y = cuda.grid(2)
    if x < src.shape[1] and y < src.shape[0]:
        r, g, b = src[y, x, 0] / 255.0, src[y, x, 1] / 255.0, src[y, x, 2] / 255.0
        hi, lo = max(r, g, b), min(r, g, b)
        delta = hi - lo
        h = 0.0
        if delta != 0:
            if hi == r:
                h = (60.0 * (g - b) / delta + 360.0) % 360.0
            elif hi == g:
                h = 60.0 * (b - r) / delta + 120.0
            else:
                h = 60.0 * (r - g) / delta + 240.0
        hue[y, x] = h
        sat[y, x] = delta / hi if hi > 0 else 0.0
        val[y, x] = hi

@cuda.jit
def hsv2rgb(hue, sat, val, dst):
    x, y = cuda.grid(2)
    if x < dst.shape[1] and y < dst.shape[0]:
        h = hue[y, x] / 60.0
        c = val[y, x] * sat[y, x]
        z = c * (1.0 - abs(h % 2.0 - 1.0))
        m = val[y, x] - c
        if h < 1: r, g, b = c, z, 0.0
        elif h < 2: r, g, b = z, c, 0.0
        elif h < 3: r, g, b = 0.0, c, z
        elif h < 4: r, g, b = 0.0, z, c
        elif h < 5: r, g, b = z, 0.0, c
        else: r, g, b = c, 0.0, z
        dst[y, x, 0] = min(255, max(0, int((r + m) * 255.0 + 0.5)))
        dst[y, x, 1] = min(255, max(0, int((g + m) * 255.0 + 0.5)))
        dst[y, x, 2] = min(255, max(0, int((b + m) * 255.0 + 0.5)))

def kuwahara_pixel(src, values, y, x, radius):
    best_var = 1e30
    best_r, best_g, best_b = 0.0, 0.0, 0.0
    count = (radius + 1) * (radius + 1)
    for q in range(4):
        y0 = y - radius if q < 2 else y
        x0 = x - radius if q % 2 == 0 else x
        total, squares = 0.0, 0.0
        sr, sg, sb = 0.0, 0.0, 0.0
        for j in range(y0, y0 + radius + 1):
            for i in range(x0, x0 + radius + 1):
                v = float64(values[j, i])
                total += v
                squares += v * v
                sr += float(src[j, i, 0])
                sg += float(src[j, i, 1])
                sb += float(src[j, i, 2])
        variance = squares / count - (total / count) ** 2
        # float32 brightness can perturb ties; use the same tolerance on CPU/GPU.
        if variance < best_var - 1e-12:
            best_var = variance
            best_r, best_g, best_b = sr / count, sg / count, sb / count
    return int(best_r), int(best_g), int(best_b)
kuwahara_device = cuda.jit(device=True)(kuwahara_pixel)
RADIUS = 2
BLOCK = (16, 16)
TILE = 16 + 2 * RADIUS

def kuwahara_cpu(src, values):
    out = src.copy()
    for y in range(RADIUS, src.shape[0] - RADIUS):
        for x in range(RADIUS, src.shape[1] - RADIUS):
            out[y, x] = kuwahara_pixel(src, values, y, x, RADIUS)
    return out

@cuda.jit
def kuwahara_global(src, values, dst):
    x, y = cuda.grid(2)
    h, w = values.shape
    if x < w and y < h:
        if RADIUS <= x < w - RADIUS and RADIUS <= y < h - RADIUS:
            r, g, b = kuwahara_device(src, values, y, x, RADIUS)
            dst[y, x, 0], dst[y, x, 1], dst[y, x, 2] = r, g, b
        else:
            for c in range(3):
                dst[y, x, c] = src[y, x, c]

@cuda.jit
def kuwahara_shared(src, values, dst):
    rgb_tile = cuda.shared.array((20, 20, 3), float32)
    v_tile = cuda.shared.array((20, 20), float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    x0 = cuda.blockIdx.x * 16 - RADIUS
    y0 = cuda.blockIdx.y * 16 - RADIUS
    tid = ty * 16 + tx
    h, w = values.shape
    # Cooperative load: gồm cả halo quanh block.
    for k in range(tid, TILE * TILE, 256):
        j, i = k // TILE, k % TILE
        sy, sx = min(h - 1, max(0, y0 + j)), min(w - 1, max(0, x0 + i))
        v_tile[j, i] = values[sy, sx]
        for c in range(3):
            rgb_tile[j, i, c] = src[sy, sx, c]
    cuda.syncthreads()
    x, y = cuda.grid(2)
    if x < w and y < h:
        if RADIUS <= x < w - RADIUS and RADIUS <= y < h - RADIUS:
            r, g, b = kuwahara_device(rgb_tile, v_tile, ty + RADIUS, tx + RADIUS, RADIUS)
            dst[y, x, 0], dst[y, x, 1], dst[y, x, 2] = r, g, b
        else:
            for c in range(3):
                dst[y, x, c] = src[y, x, c]


In [ ]:
d_src = cuda.to_device(image)
d_h, d_s, d_v = [cuda.device_array((H, W), dtype=np.float32) for _ in range(3)]
hsv_s = gpu_time(lambda: rgb2hsv[grid(BLOCK), BLOCK](d_src, d_h, d_s, d_v))
values = d_v.copy_to_host()
assert np.allclose(values, image.max(axis=2) / 255.0, atol=1e-6)
reference, cpu_s = cpu_time(lambda: kuwahara_cpu(image, values))
d_dst = cuda.device_array_like(d_src)
rows, outputs = [], {}
times = {}
for name, kernel in [('global', kuwahara_global), ('shared', kuwahara_shared)]:
    seconds = gpu_time(lambda: kernel[grid(BLOCK), BLOCK](d_src, d_v, d_dst))
    result = d_dst.copy_to_host()
    error = int(np.abs(result.astype(int) - reference.astype(int)).max())
    assert error <= 1
    outputs[name], times[name] = result, seconds
    rows.append(dict(mode=name, cpu_ms=cpu_s * 1000, gpu_ms=seconds * 1000,
                     speedup_cpu=cpu_s / seconds, max_error=error))
ratio = times['global'] / times['shared']
for row in rows:
    row['global_over_shared'] = ratio
    row['rgb2hsv_ms'] = hsv_s * 1000
print('Global / shared speedup:', ratio)
save_image('image_kuwahara_cpu.png', reference)
for name, result in outputs.items():
    save_image('image_kuwahara_' + name + '.png', result)
show_images([('Input', image), ('Global', outputs['global']), ('Shared', outputs['shared'])])
plt.figure(figsize=(6, 3))
plt.bar(['CPU', 'GPU global', 'GPU shared'], [cpu_s * 1000, times['global'] * 1000, times['shared'] * 1000])
plt.yscale('log'); plt.ylabel('Time (ms, log scale)')
plt.tight_layout(); plt.savefig(OUT / 'performance.png', dpi=120); plt.show()


In [ ]:
report_text = r"""
\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage[T5]{fontenc}
\usepackage{graphicx}
\usepackage[margin=1.5cm]{geometry}
\title{Lab 10: Kuwahara Filter}
\author{Nguyễn Thanh Hùng -- 2540056}
\date{}
\begin{document}
\maketitle
RGB2HSV produces brightness $V=\max(R,G,B)/255$. For radius 2, four $3\times3$ windows are compared by V variance; the RGB mean of the lowest-variance window becomes the output.
The CPU and global-memory GPU compute the same windows. The shared GPU cooperatively loads a $20\times20$ RGB/V tile including halo for a $16\times16$ block, then synchronizes. The two-pixel border is preserved.
The notebook checks both GPU outputs and reports CPU/global and global/shared speedups. RGB2HSV time is separate.
Neighboring x threads access adjacent pixels; shared tiles reuse overlapping neighborhoods. No claim of eliminated bank conflicts is made without profiling.

Kernel timing uses \texttt{time.time()} after JIT warm-up and CUDA synchronization,
averaged over three GPU runs. Device allocation and host/device copies are excluded.
CPU measurements are single runs. Speedup is relative to the stated CPU implementation.
Actual measurements appear after running the notebook on Colab; no timings are assumed here.
\IfFileExists{results.tex}{\input{results.tex}}{}
\IfFileExists{performance.png}{\begin{center}\includegraphics[width=.75\linewidth]{performance.png}\end{center}}{}
\IfFileExists{images.png}{\begin{center}\includegraphics[width=\linewidth]{images.png}\end{center}}{}
\IfFileExists{histograms.png}{\begin{center}\includegraphics[width=.8\linewidth]{histograms.png}\end{center}}{}
\end{document}
"""
save_results(rows, 'Report.10.kuwahara.tex', report_text)
